# Python II — Modules and Packages
### Demo notebook — Lesson 3

In Lesson 1 we built the **animal shelter** (`Animal`, `Dog`, `Cat`, `Shelter`), and in Lesson 2 we learned how to
**raise**, **catch** and **warn**. But all that code lives *inside a notebook*: to reuse it somewhere else we would have to copy and paste it.

Today we take that code out of the notebook and turn it into something **reusable and installable**, in three steps:

1. a **module** → one `.py` file we can `import`;
2. a **package** → a folder of modules that work together;
3. an **installable distribution** → something anyone can install with `pip install`, exactly like `numpy` or `pandas`.

| # | Section | What we do |
|---|---------|------------|
| 1 | Modules everywhere | look at modules we already use |
| 2 | Our first module | save the Lesson 1 classes in `animals.py` and import them |
| 3 | What happens when you `import` | code runs once, the module cache, `reload`, `if __name__ == "__main__"` |
| 4 | Where Python looks for modules | `sys.path`, `ModuleNotFoundError` |
| 5 | From module to package | a folder, `__init__.py`, relative imports, custom exceptions |
| 6 | The package's front door | `__init__.py`, `__all__`, `__version__` |
| 7 | Using our package | shelter + exceptions + warnings (Lessons 1 and 2 together) |
| 8 | A command-line program | `cli.py` and `__main__.py` |
| 9 | Making it installable | project layout and `pyproject.toml` |
| 10 | Installing in *editable* mode | `pip install -e` |
| 11 | Building and installing a wheel | `python -m build`, the `.whl` file, `pip install` |
| 12 | Sharing and cleaning up | how others can install it |

⚠️ **Before you start.** This notebook **creates files** in the folder where it is saved
(`animals.py`, `animals.csv` and a folder `shelter_project/`). Put it in a folder of its own.

💡 To write files we use **`%%writefile <path>`**: a Jupyter *magic* that, placed on the first line of a cell,
**saves the rest of the cell into a file** instead of running it.

---
## 1. Modules everywhere

A **module** is simply a file containing Python code (`something.py`). We have been using modules since day one:
every `import` loads one.

| Term | What it is | Examples |
|------|-----------|----------|
| **module** | one `.py` file | `warnings`, `random`, our `animals.py` |
| **package** | a folder of modules (with an `__init__.py` file) | `json`, `numpy`, our `animal_shelter/` |
| **distribution** | an archive that `pip` can install, containing one or more packages | `numpy-2.1.0-...whl` |

Every module is an **object** (remember: in Python *everything* is an object), with attributes like `__name__` and `__file__`.

In [ ]:
import json
import warnings

print(warnings.__name__, "->", warnings.__file__)   # a single file: warnings.py
print(json.__name__, "   ->", json.__file__)        # a package: notice the __init__.py at the end

---
## 2. Our first module: `animals.py`

We save the final classes of Lesson 1 into a file called `animals.py`, next to this notebook.

Two lines are new, and only there **for the demo**:
- a `print(...)` at the top, so we can *see* when the code of the module is executed;
- an `if __name__ == "__main__":` block at the bottom (explained in section 3).

In [ ]:
%%writefile animals.py
# good practice: create docstrings for modules, classes, and functions
"""animals.py: the animal classes of Lesson 1, saved in a file.            

A module is simply a .py file: everything defined at its top level
(classes, functions, variables) can be imported by other code.
"""

print(f"Executing the code of module {__name__!r} ...")   # demo only: shows WHEN this file runs

KINGDOM = "Animalia"    # a module-level variable (a constant: by convention in UPPERCASE)


class Animal:
    """A generic animal with a name and an age."""

    count = 0

    def __init__(self, name, age):
        self.name = name
        self.age = age               # goes through the property setter below
        Animal.count += 1

    @property
    def age(self):
        return self._age

    @age.setter
    def age(self, value):
        if not Animal.is_valid_age(value):
            raise ValueError(f"Invalid age: {value!r}")
        self._age = value

    @staticmethod
    def is_valid_age(value):
        return isinstance(value, int) and value >= 0

    def speak(self):
        raise NotImplementedError(f"{type(self).__name__} must implement speak()")

    def introduce(self):
        print(f"I am {self}, and I say: {self.speak()}")

    def birthday(self):
        self.age += 1

    def __repr__(self):
        return f"{type(self).__name__}({self.name!r}, {self.age!r})"

    def __str__(self):
        return f"{self.name} ({self.age} y.o.)"

    def __eq__(self, other):
        if not isinstance(other, Animal):
            return NotImplemented
        return (type(self), self.name, self.age) == (type(other), other.name, other.age)

    def __lt__(self, other):
        if not isinstance(other, Animal):
            return NotImplemented
        return self.age < other.age


class Mammal(Animal):
    is_hairy = True


class Dog(Mammal):
    def __init__(self, name, age, breed="unknown"):
        super().__init__(name, age)
        self.breed = breed

    def speak(self):
        return "Woof!"

    def __repr__(self):
        return f"Dog({self.name!r}, {self.age!r}, {self.breed!r})"


class Cat(Mammal):
    def speak(self):
        return "Meow!"


def oldest(animals):
    """Return the oldest animal of a list (it relies on Animal.__lt__)."""
    return max(animals)


if __name__ == "__main__":
    # This block runs ONLY when the file is executed as a script ("python animals.py"),
    # NOT when the file is imported. Perfect for quick tests and demos.
    rex = Dog("Rex", 3, "Labrador")
    rex.introduce()

### 2.1 `import`
`import animals` looks for a file called `animals.py`, **executes it**, and gives us a module object called `animals`.
Everything defined in the file is reached with the **dot operator**, as with attributes of an object.

In [ ]:
import animals

rex = animals.Dog("Rex", 3, "Labrador")
luna = animals.Cat("Luna", 5)

rex.introduce()
print(animals.KINGDOM)
print("Oldest:", animals.oldest([rex, luna]))

### 2.2 Different ways to import

| You write | You get | Use it like |
|-----------|---------|-------------|
| `import animals` | the module | `animals.Dog(...)` |
| `import animals as an` | the module, with a shorter name | `an.Dog(...)` (as in `import numpy as np`) |
| `from animals import Dog, Cat` | only those names | `Dog(...)` |
| `from animals import *` | **every** public name | ⚠️ avoid: you can't tell where names come from, and they may overwrite yours |

Whatever style you use, the module is loaded only once: they all refer to the **same objects**.

In [ ]:
from animals import Dog, Cat
import animals as an

tom = Cat("Tom", 1)
print(tom)
print(an.Dog is Dog is animals.Dog)    # True: the very same class object

### 2.3 A module is an object
Like any object, a module has attributes. Note that the **docstring** at the top of the file becomes `animals.__doc__`,
and that's what `help(animals)` shows: always document your modules!

In [ ]:
print(type(animals))
print(animals.__name__)
print(animals.__file__)
print(animals.__doc__)
print(help(animals))   # shows the module's docstring, its classes, and functions

---
## 3. What happens when you `import`

### 3.1 The code runs only ONCE
After the first time, Python stores the module in a dictionary, **`sys.modules`** (the *module cache*),
and every later `import` simply returns the stored object. Importing is fast, and the code is not executed again.

In [ ]:
import sys

import animals                          # nothing is printed this time
print("animals" in sys.modules)         # True: it's in the cache

### 3.2 ⚠️ The notebook pitfall: editing a module that is already imported
Because of the cache, **if you change the file, the notebook does not see the change**.
Let's add a new function at the end of `animals.py` (`%%writefile -a` **appends** to the file instead of overwriting it).

In [ ]:
%%writefile -a animals.py


def youngest(animals):
    """Return the youngest animal of a list (appended at the end, just for the demo)."""
    return min(animals)

In [ ]:
import animals          # the cached (old) version is returned: the file is NOT read again

try:
    animals.youngest([rex, luna])
except AttributeError as err:
    print("AttributeError:", err)

The function **is** in the file, but our module object is the old one. Safest solution to solve this problem is to **restar the kernel**!

### 3.3 `if __name__ == "__main__":`
Every module has a `__name__` attribute:
- when the file is **imported**, `__name__` is the name of the module (`"animals"`);
- when the file is **run directly** as a program (`python animals.py`), `__name__` is the special string `"__main__"`.

So the code under `if __name__ == "__main__":` runs **only when the file is executed as a script**, never on import.
It's the place for quick tests or for starting a program. (The notebook itself is the "main" program.)

In [ ]:
print("Name of the imported module:", animals.__name__)
print("Name of this notebook:      ", __name__)

Now let's run `animals.py` as a script. In a notebook, `!` runs a **terminal command**.

In [ ]:
!python animals.py

This time the message says `'__main__'`: the file was run as a script, so the test block ran and Rex introduced himself.

---
## 4. Where does Python look for modules?

When you write `import animals`, Python searches a list of folders, **in order**, and uses the first match.
That list is **`sys.path`**:
1. the folder of the script / notebook that is running (that's why `import animals` worked!);
2. the folders of the **standard library** (`json`, `warnings`, ...);
3. **`site-packages`**: the folder where `pip install` puts the packages you install (keep this in mind for section 10).

In [ ]:
for folder in sys.path:
    print(folder)

If the module is not found in any of these folders, we get a `ModuleNotFoundError` (a child class of `ImportError`).
You have met it every time you forgot to `pip install` something!

In [ ]:
try:
    import shelter_tools          # no such file anywhere in sys.path
except ModuleNotFoundError as err:
    print("ModuleNotFoundError:", err)

---
## 5. From module to package

One file is fine for a few classes. But our project keeps growing: animals, the shelter, our custom exceptions,
code to read animals from a file, a command-line program... Putting everything in one huge file makes it hard to read and to maintain.

A **package** is a **folder of modules** that belong together. Python recognises it as a package thanks to a
special file inside it: **`__init__.py`**.

This is the project we are going to build. Today we create every file directly in its **final** place
(`pyproject.toml`, `README.md` and the `src/` folder will be explained in section 9):

```
shelter_project/                  <- the PROJECT folder: what we share with others
├── pyproject.toml                <- section 9: tells pip how to install the project
├── README.md                     <- section 9
└── src/
    └── animal_shelter/           <- the PACKAGE: what we import
        ├── __init__.py           <- makes the folder a package (section 6)
        ├── exceptions.py         <- our custom exceptions (Lesson 2)
        ├── animals.py            <- Animal, Mammal, Dog, Cat (Lesson 1)
        ├── shelter.py            <- the Shelter container (Lesson 1)
        ├── loaders.py            <- read animals from a CSV file (Lesson 2: warnings!)
        ├── cli.py                <- section 8: a command-line program
        └── __main__.py           <- section 8: lets us run "python -m animal_shelter"
```

**Naming rules.** A package name must be a valid Python name: lowercase, words separated by underscores.
No hyphens: `import animal-shelter` would mean *"animal minus shelter"*!

In [ ]:
from pathlib import Path

PACKAGE_DIR = Path("shelter_project") / "src" / "animal_shelter"
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)      # creates all the missing folders

print("Package folder:", PACKAGE_DIR.resolve())

### 5.1 `__init__.py`
For now it only contains a docstring (the documentation of the whole package). We will come back to it in section 6.

In [ ]:
%%writefile shelter_project/src/animal_shelter/__init__.py
"""animal_shelter: manage the animals of a shelter (Python II demo package)."""

### 5.2 `exceptions.py`: the errors of our package
In Lesson 2 we created custom exceptions by inheriting from `Exception`. A good practice for packages is to create
**one base exception** for the whole package, and make all the others inherit from it:

```
Exception
└── ShelterError                 <- "something went wrong in animal_shelter"
    ├── InvalidAgeError
    ├── ShelterFullError
    └── AnimalNotFoundError
```

This way the user of our package can catch one specific error (`except ShelterFullError`) **or** any error coming
from our package with a single `except ShelterError`.

In [ ]:
%%writefile shelter_project/src/animal_shelter/exceptions.py
"""Custom exceptions of the animal_shelter package.

They all inherit from ShelterError, so a user of the package can catch
*every* error coming from our code with a single `except ShelterError`.
"""


class ShelterError(Exception):
    """Base class for all the errors raised by animal_shelter."""


class InvalidAgeError(ShelterError):
    """Raised when an age is not a non-negative integer."""


class ShelterFullError(ShelterError):
    """Raised when admitting an animal into a shelter that has no free places."""


class AnimalNotFoundError(ShelterError):
    """Raised when looking for an animal that is not in the shelter."""

### 5.3 `animals.py`: importing from a sibling module
The animal classes are the same as before, with two differences:
- no `print` and no `if __name__ == "__main__":` block (modules inside a package are meant to be imported);
- an invalid age now raises **our** `InvalidAgeError`, which lives in `exceptions.py`.

To use it we write **`from .exceptions import InvalidAgeError`**. The leading dot is a **relative import**:
*"the module `exceptions` in **this same package**"*.

In [ ]:
%%writefile shelter_project/src/animal_shelter/animals.py
"""The animals that can live in a shelter."""

from .exceptions import InvalidAgeError    # relative import: "exceptions.py in this same package"


class Animal:
    """A generic animal with a name and an age. Subclasses must implement speak()."""

    def __init__(self, name, age):
        self.name = name
        self.age = age               # goes through the property setter below

    @property
    def age(self):
        return self._age

    @age.setter
    def age(self, value):
        if not Animal.is_valid_age(value):
            raise InvalidAgeError(f"Invalid age for {self.name!r}: {value!r}")
        self._age = value

    @staticmethod
    def is_valid_age(value):
        """True if value is a non-negative integer."""
        return isinstance(value, int) and value >= 0

    def speak(self):
        raise NotImplementedError(f"{type(self).__name__} must implement speak()")

    def introduce(self):
        print(f"I am {self}, and I say: {self.speak()}")

    def birthday(self):
        self.age += 1

    def __repr__(self):
        return f"{type(self).__name__}({self.name!r}, {self.age!r})"

    def __str__(self):
        return f"{self.name} ({self.age} y.o.)"

    def __eq__(self, other):
        if not isinstance(other, Animal):
            return NotImplemented
        return (type(self), self.name, self.age) == (type(other), other.name, other.age)

    def __lt__(self, other):
        if not isinstance(other, Animal):
            return NotImplemented
        return self.age < other.age


class Mammal(Animal):
    """An animal with hair."""

    is_hairy = True


class Dog(Mammal):
    """A dog, with an optional breed."""

    def __init__(self, name, age, breed="unknown"):
        super().__init__(name, age)
        self.breed = breed

    def speak(self):
        return "Woof!"

    def __repr__(self):
        return f"Dog({self.name!r}, {self.age!r}, {self.breed!r})"


class Cat(Mammal):
    """A cat."""

    def speak(self):
        return "Meow!"

### 5.4 `shelter.py`
The `Shelter` container of Lesson 1, improved with what we learned in Lesson 2. It now:
- refuses objects that are not animals (`TypeError`);
- has an optional `capacity` and raises `ShelterFullError` when it's full;
- has an `adopt(name)` method (Exercise 2 of Lesson 1!) that raises `AnimalNotFoundError` when the animal is not there.

It needs classes from **two** sibling modules, so it has two relative imports.

In [ ]:
%%writefile shelter_project/src/animal_shelter/shelter.py
"""The Shelter class: a container of animals."""

from .animals import Animal
from .exceptions import AnimalNotFoundError, ShelterFullError


class Shelter:
    """A shelter holding animals, with an optional maximum capacity (None = no limit)."""

    def __init__(self, name, animals=None, capacity=None):
        self.name = name
        self.capacity = capacity
        self.animals = []
        for animal in animals or []:      # we use admit(), so the checks are applied here too
            self.admit(animal)

    def admit(self, animal):
        """Add an animal to the shelter."""
        if not isinstance(animal, Animal):
            raise TypeError(f"Only animals can be admitted, got {type(animal).__name__}")
        if self.capacity is not None and len(self) >= self.capacity:
            raise ShelterFullError(f"Shelter {self.name!r} is full ({self.capacity} places)")
        self.animals.append(animal)

    def adopt(self, name):
        """Remove the animal called `name` from the shelter and return it."""
        for i, animal in enumerate(self.animals):
            if animal.name == name:
                return self.animals.pop(i)
        raise AnimalNotFoundError(f"No animal called {name!r} in shelter {self.name!r}")

    def __repr__(self):
        return f"Shelter({self.name!r}, {self.animals!r})"

    def __len__(self):
        return len(self.animals)

    def __getitem__(self, index):
        return self.animals[index]

    def __contains__(self, animal):
        return animal in self.animals

    def __add__(self, other):
        if not isinstance(other, Shelter):
            return NotImplemented
        return Shelter(f"{self.name} & {other.name}", self.animals + other.animals)

### 5.5 `loaders.py`: reading animals from a file
This module reads a CSV file (one animal per line) and is basically the `read_grades` example of Lesson 2:
- an invalid line is **skipped with a warning**: the rest of the file is still usable;
- if the file does not exist we do **not** catch `FileNotFoundError`: we let it reach whoever called the function.

It also uses two modules of the **standard library**, `csv` and `warnings`: they are imported normally (absolute imports),
while our own modules are imported with the dot. By convention, standard library imports come first.

The function `_make_animal` starts with an underscore: by convention it's **internal** to the module, not meant to be used from outside.

In [ ]:
%%writefile shelter_project/src/animal_shelter/loaders.py
"""Read animals from a CSV file.

Expected format: a header line, then one animal per line.

    kind,name,age,breed
    dog,Rex,3,Labrador
    cat,Luna,5,
"""

import csv          # standard library imports first...
import warnings

from .animals import Cat, Dog                  # ...then the modules of our package
from .exceptions import InvalidAgeError


def load_animals(path):
    """Return the list of animals described in the CSV file at `path`.

    Invalid lines are skipped with a warning. If the file does not exist,
    FileNotFoundError is raised: the caller decides what to do about it.
    """
    animals = []
    with open(path, newline="", encoding="utf-8") as f:
        for line_number, row in enumerate(csv.DictReader(f), start=2):   # line 1 is the header
            try:
                animals.append(_make_animal(row))
            except (ValueError, InvalidAgeError) as err:
                warnings.warn(f"{path}, line {line_number} skipped: {err}")
    return animals


def _make_animal(row):
    """Build one animal from a CSV row (a dict). Internal helper."""
    kind = (row.get("kind") or "").strip().lower()
    name = (row.get("name") or "").strip()
    if not name:
        raise ValueError("missing name")
    age = int(row.get("age") or "")             # ValueError if missing or not a number

    if kind == "dog":
        breed = (row.get("breed") or "").strip() or "unknown"
        return Dog(name, age, breed)
    if kind == "cat":
        return Cat(name, age)
    raise ValueError(f"unknown kind of animal {kind!r}")

### 5.6 Relative and absolute imports

Inside a package there are two ways to import a sibling module:

| Absolute import | Relative import | Meaning |
|-----------------|-----------------|---------|
| `from animal_shelter.exceptions import ShelterError` | `from .exceptions import ShelterError` | `exceptions.py` in the same package |
| `from animal_shelter import Shelter` | `from . import Shelter` | the package itself (its `__init__.py`) |

Both work. Relative imports are shorter and keep working if the package is renamed. They only work **inside a package**:
that's why you can't run a module of a package as a script (`python shelter.py` gives
*"ImportError: attempted relative import with no known parent package"*).

### 5.7 Trying the package (with a temporary hack)
To import `animal_shelter`, its parent folder (`shelter_project/src`) must be in `sys.path`, and right now it isn't.
As a **temporary hack** we add it by hand. It works, but only in this notebook, and only as long as
the folder doesn't move: in section 10 we will replace this with a proper installation.

In [ ]:
import sys

SRC = str(Path("shelter_project/src").resolve())
if SRC not in sys.path:
    sys.path.insert(0, SRC)       # ⚠️ temporary hack, replaced by "pip install" in section 10

from animal_shelter.animals import Dog, Cat
from animal_shelter.shelter import Shelter
from animal_shelter.exceptions import ShelterFullError

north = Shelter("North", [Dog("Rex", 3, "Labrador"), Cat("Luna", 5)], capacity=2)
print(north)

try:
    north.admit(Cat("Tom", 1))
except ShelterFullError as err:
    print("ShelterFullError:", err)

It works, but the user must know **in which file** each class lives (`animal_shelter.animals`, `animal_shelter.shelter`, ...).
The package itself is still empty:

In [ ]:
import animal_shelter

print([name for name in dir(animal_shelter) if not name.startswith("_")])   # only the submodules we imported

try:
    animal_shelter.load_animals
except AttributeError as err:
    print("AttributeError:", err)

---
## 6. The package's front door: `__init__.py`

`__init__.py` is executed when the package is imported, and **whatever it defines or imports becomes an attribute of the package**.
We use it to choose the **public interface**: the names a user needs, available directly as `from animal_shelter import ...`,
without knowing how the files are organised inside. That's why you write `import numpy as np; np.array(...)` and not
`numpy.core.multiarray.array`.

We also add:
- **`__version__`**: the version of the package (a widespread convention);
- **`__all__`**: the list of public names, i.e. what `from animal_shelter import *` imports.

In [ ]:
%%writefile shelter_project/src/animal_shelter/__init__.py
"""animal_shelter: manage the animals of a shelter (Python II demo package).

Quick start:

    from animal_shelter import Dog, Cat, Shelter

    north = Shelter("North", [Dog("Rex", 3, "Labrador"), Cat("Luna", 5)])
"""

from .animals import Animal, Cat, Dog, Mammal
from .exceptions import AnimalNotFoundError, InvalidAgeError, ShelterError, ShelterFullError
from .loaders import load_animals
from .shelter import Shelter

__version__ = "0.1.0"

__all__ = [
    "Animal", "Mammal", "Dog", "Cat",
    "Shelter",
    "load_animals",
    "ShelterError", "InvalidAgeError", "ShelterFullError", "AnimalNotFoundError",
]

Remember section 3: the package is **already imported**, so we must `reload` it to see the new `__init__.py`.

In [ ]:
import importlib    # alternative way to kernel restart
importlib.reload(animal_shelter)

print(animal_shelter.__version__)
print(animal_shelter.__all__)
print(animal_shelter.Shelter)                                # same class as animal_shelter.shelter.Shelter
print(animal_shelter.Shelter is animal_shelter.shelter.Shelter)

---
## 7. Using our package

From now on we use the package **as its users would**: through the front door, without caring about the files inside.
Here everything we learned in Lessons 1 and 2 works together.

In [ ]:
from animal_shelter import Cat, Dog, Shelter

north = Shelter("North", [Dog("Rex", 3, "Labrador"), Cat("Luna", 5)], capacity=3)
north.admit(Cat("Tom", 1))
print(north, "\n")

for animal in sorted(north):         # sorted by age, thanks to __lt__
    animal.introduce()

### 7.1 Catching the errors of the package
We can catch one specific error...

In [ ]:
from animal_shelter import AnimalNotFoundError, ShelterError, ShelterFullError

try:
    north.admit(Dog("Fido", 8, "Beagle"))
except ShelterFullError as err:
    print("ShelterFullError:", err)

try:
    north.adopt("Garfield")
except AnimalNotFoundError as err:
    print("AnimalNotFoundError:", err)

luna = north.adopt("Luna")
print(f"{luna.name} has been adopted! Animals left: {len(north)}")

...or **any** error coming from our package, thanks to the common base class `ShelterError`:

In [ ]:
for name, age in [("Ghost", -1), ("Shadow", "two"), ("Buddy", 4)]:
    try:
        dog = Dog(name, age)
    except ShelterError as err:          # InvalidAgeError is a ShelterError
        print(f"{type(err).__name__}: {err}")
    else:
        print("Created:", repr(dog))

### 7.2 Loading animals from a file
Let's create a CSV file with a few **invalid lines** on purpose: a wrong age, an unknown kind of animal, a negative age.

In [ ]:
%%writefile animals.csv
kind,name,age,breed
dog,Rex,3,Labrador
cat,Luna,5,
dog,Fido,8,Beagle
cat,Tom,one,
fish,Nemo,1,
dog,Ghost,-1,Husky
cat,Milo,2,

The invalid lines produce **warnings**, but the program continues with the valid ones.
A missing file, instead, is an **error**: the package raises it, and *we* (the application) decide how to handle it.

In [ ]:
from animal_shelter import load_animals

south = Shelter("South", load_animals("animals.csv"))
print(f"\n{len(south)} animals loaded:", south.animals)

try:
    load_animals("missing.csv")
except FileNotFoundError as err:
    print("We decide what to do:", err)

---
## 8. A command-line program: `cli.py` and `__main__.py`

Packages often come with a small program you can run from the terminal (think of `pip` or `jupyter` themselves!).
Ours prints a report of the animals listed in a CSV file.

- `cli.py` (*command-line interface*) contains a function `main()`. The words typed after the program name arrive in the list
  **`sys.argv`** (`sys.argv[0]` is the program itself). `main()` returns an **exit code**: `0` means success, anything else means error.
  Here, unlike in a library function, catching `FileNotFoundError` and printing a friendly message **is** the right thing to do:
  this is the *application*.
- `__main__.py` is a special file: it's what runs when you execute **`python -m animal_shelter`**
  (same idea as `if __name__ == "__main__":`, but for a whole package).

In [ ]:
%%writefile shelter_project/src/animal_shelter/cli.py
"""Command-line program: print a report of the animals listed in a CSV file.

After installing the package, from a terminal:

    shelter-report animals.csv
    python -m animal_shelter animals.csv      (equivalent)
"""

import sys

from . import __version__
from .loaders import load_animals
from .shelter import Shelter


def main(argv=None):
    """Entry point of the program. Returns the exit code (0 = success)."""
    if argv is None:
        argv = sys.argv[1:]            # the words typed after the program name

    if len(argv) != 1:
        print("usage: shelter-report FILE.csv", file=sys.stderr)
        return 2

    path = argv[0]
    try:
        shelter = Shelter(path, load_animals(path))
    except FileNotFoundError:
        print(f"error: file {path!r} not found", file=sys.stderr)
        return 1

    print(f"animal_shelter {__version__}: {len(shelter)} animals in {path}")
    for animal in sorted(shelter):
        animal.introduce()
    return 0

In [ ]:
%%writefile shelter_project/src/animal_shelter/__main__.py
"""Executed by:  python -m animal_shelter FILE.csv"""

import sys

from .cli import main

sys.exit(main())     # the return value of main() becomes the exit code of the program

Since `main()` accepts the arguments as a list, we can also test it directly from the notebook:

In [ ]:
from animal_shelter.cli import main

print("exit code:", main(["animals.csv"]), "\n")
print("exit code:", main(["missing.csv"]))

---
## 9. Making it installable

Our package works, but only thanks to the `sys.path` hack: it only works in this notebook, with this folder structure.
To use it from **any** folder, any script, any notebook (or on a colleague's computer) we must **install** it with `pip`,
like any other package.

`pip` needs a file that describes the project: **`pyproject.toml`**, placed in the project folder.
It's the *identity card* of the project, in [TOML](https://toml.io) format (sections in `[brackets]`, then `key = value`):

- **`[build-system]`**: which tool `pip` must use to *build* the package. We use `setuptools`, the classic one.
- **`[project]`**: name, version, description, required Python version, and **dependencies** (other packages that pip must install
  together with ours; we have none, but e.g. `dependencies = ["numpy>=1.24"]`).
- **`[project.scripts]`**: command-line programs to create at install time. `shelter-report = "animal_shelter.cli:main"` means:
  *create a command `shelter-report` that calls the function `main` of the module `animal_shelter.cli`*.

In [ ]:
%%writefile shelter_project/pyproject.toml
# pyproject.toml: the "identity card" of the project, read by pip

[build-system]                        # HOW to build the package
requires = ["setuptools>=64"]
build-backend = "setuptools.build_meta"

[project]                             # WHAT the package is
name = "animal-shelter"               # the name used by pip:  pip install animal-shelter
version = "0.1.0"                     # keep it equal to __version__ in __init__.py!
description = "Manage the animals of a shelter (Python II demo package)"
readme = "README.md"
requires-python = ">=3.9"
authors = [{ name = "Python II students" }]
dependencies = []                     # packages ours needs, e.g. ["numpy>=1.24"]

[project.scripts]                     # command-line programs created when installing
shelter-report = "animal_shelter.cli:main"

In [ ]:
%%writefile shelter_project/README.md
# animal-shelter

A tiny package to manage the animals of a shelter, built during the Python II course.

## Install

    pip install .

## Use

    from animal_shelter import Dog, Cat, Shelter

    north = Shelter("North", [Dog("Rex", 3, "Labrador"), Cat("Luna", 5)])

or, from a terminal:

    shelter-report animals.csv

### 9.1 Two different names
Note that the project has **two names**, and that's normal:

| | Our project | Other examples |
|--|--|--|
| **distribution name** (for `pip install`) | `animal-shelter` | `scikit-learn`, `pillow`, `beautifulsoup4` |
| **import name** (the package folder) | `animal_shelter` | `sklearn`, `PIL`, `bs4` |

### 9.2 Why the `src/` folder?
The package lives in `shelter_project/src/` and not directly in `shelter_project/`. This is called the **src layout**:
since `src/` is not in `sys.path`, you **cannot** import the package by accident just because you are working in that folder.
You are forced to install it, so you always test **exactly what your users will get**.

Here is our complete project:

In [ ]:
def show_tree(folder, indent=""):
    """Print the content of a folder as a tree (skipping caches and build leftovers)."""
    skip = {"__pycache__", "build", "dist"}
    items = sorted(p for p in Path(folder).iterdir()
                   if p.name not in skip and not p.name.endswith(".egg-info"))
    for i, item in enumerate(items):
        last = (i == len(items) - 1)
        print(indent + ("└── " if last else "├── ") + item.name + ("/" if item.is_dir() else ""))
        if item.is_dir():
            show_tree(item, indent + ("    " if last else "│   "))

print("shelter_project/")
show_tree("shelter_project")

### ⚠️ Restart the kernel now!
**Kernel → Restart** (in Colab: *Runtime → Restart session*). This throws away the `sys.path` hack and every imported module,
so from now on we can prove that the package works **without any trick**.

After restarting, continue from the next cell (don't re-run the cells above).

---
## 10. Installing in *editable* mode

After the restart, the hack is gone: Python doesn't know where our package is.

In [ ]:
import subprocess
import sys
import tempfile
from pathlib import Path

try:
    import animal_shelter
except ModuleNotFoundError as err:
    print("ModuleNotFoundError:", err)
    print("Correct! It's not installed, and shelter_project/src is not in sys.path.")

### 10.1 A small helper: running Python somewhere else
To prove that the package works **from anywhere**, we will run small pieces of code in a **brand-new Python process**,
started in a completely different folder (the temporary folder of the computer). This is exactly what happens when a
colleague imports your package from their own script.

`subprocess.run` starts a program and waits for it to finish; `capture_output=True, text=True` collects what it prints, so we can show it here.

In [ ]:
def run(command, folder=tempfile.gettempdir()):
    """Run a command in a NEW process, inside `folder`, and print its output."""
    result = subprocess.run(command, cwd=folder, capture_output=True, text=True)
    print(result.stdout + result.stderr)


def run_python(code):
    """Run some Python code in a NEW Python process, far away from this notebook."""
    run([sys.executable, "-c", code])


run_python("import os; print('Hello from', os.getcwd())")
run_python("import animal_shelter")

### 10.2 `pip install -e`
Now we install the project. We use **`%pip`** (with `%`, not `!`): this magic makes sure the package is installed
in the **same Python environment used by this notebook**.

The **`-e`** option (*editable*) is the developer mode: pip does **not** copy our files, it only registers a *link*
to the `src` folder. So every change we make to the code is immediately visible, without re-installing.
This is how you install a package **while you are developing it**.

💡 **Virtual environments.** In real projects, each project gets its own isolated environment
(`python -m venv .venv`, or `conda create -n myproject`), so that packages installed for one project don't interfere
with the others. Today, to keep things simple, we install into the notebook's environment and uninstall at the end (section 12).

In [ ]:
%pip install -e ./shelter_project

Now the package can be imported from **any** folder: no more hacks.
`__file__` shows that it still points to **our** `src` folder: that's the editable link.

In [ ]:
run_python("import animal_shelter; print(animal_shelter.__version__, animal_shelter.__file__)")

run_python("""
from animal_shelter import Dog, Shelter
shelter = Shelter("Far away", [Dog("Rex", 3, "Labrador")])
shelter[0].introduce()
""")

`pip` now knows our project, exactly like any package downloaded from the internet:

In [ ]:
%pip show animal-shelter

### 10.3 Editable means: changes are visible immediately
Let's change what cats say directly in the source file, run the code again (no re-install!), and then restore the file.

In [ ]:
animals_file = Path("shelter_project/src/animal_shelter/animals.py")
original = animals_file.read_text(encoding="utf-8")

animals_file.write_text(original.replace('return "Meow!"', 'return "Purrrr..."'), encoding="utf-8")
run_python("from animal_shelter import Cat; Cat('Luna', 5).introduce()")    # the change is already there

animals_file.write_text(original, encoding="utf-8")                         # put the original code back
run_python("from animal_shelter import Cat; Cat('Luna', 5).introduce()")

💡 **What about this notebook?** A package installed *while the kernel is running* becomes importable here only after a
**kernel restart** (and, as we saw in section 3, after editing the code you need a restart or `importlib.reload` too).
That's why in these sections we test it in new processes instead.

---
## 11. Building and installing a wheel

The editable install is perfect **for us**, while we develop. But to give the package to **someone else**, we need a single file
containing everything: a **distribution**. There are two kinds:

| File | Name | What it contains |
|------|------|------------------|
| `animal_shelter-0.1.0-py3-none-any.whl` | **wheel** | the package, ready to install: pip just unpacks it |
| `animal_shelter-0.1.0.tar.gz` | **sdist** (*source distribution*) | the source code of the project, pip must build it first |

To build them we use the official tool **`build`** (install it once):

In [ ]:
%pip install build

`python -m build <project folder>` reads `pyproject.toml`, builds the package and saves the results in a new folder `dist/`.
It prints a lot of output: the important part is the last line, *"Successfully built ..."*.

In [ ]:
!"{sys.executable}" -m build shelter_project

In [ ]:
for file in sorted(Path("shelter_project/dist").iterdir()):
    print(f"{file.name:45} {file.stat().st_size:>7} bytes")

### 11.1 What's inside a wheel?
A wheel is just a **zip file** with a special name. Its name tells pip where it can be installed:

`animal_shelter` - `0.1.0` - `py3` - `none` - `any` `.whl` → *name - version - Python 3 - no compiled code - any operating system*

Packages with compiled code (like `numpy`) have one wheel per operating system, e.g. `...-cp312-cp312-win_amd64.whl`.
Let's look inside ours: our modules, plus a `.dist-info` folder with the *metadata* coming from `pyproject.toml`.

In [ ]:
import zipfile

wheel = next(Path("shelter_project/dist").glob("*.whl"))
print(wheel.name)
with zipfile.ZipFile(wheel) as zf:
    for name in zf.namelist():
        print("   ", name)

### 11.2 Installing the wheel
This is what your colleague would do with the `.whl` file you sent them. First we remove the editable install, then we install the wheel.

This time pip **copies** the files into `site-packages` (remember `sys.path`, section 4): the installed package
no longer depends on our project folder. If we now changed `shelter_project/src`, the installed version would **not** change.

In [ ]:
%pip uninstall -y animal-shelter

In [ ]:
%pip install {wheel}

In [ ]:
run_python("import animal_shelter; print(animal_shelter.__version__, animal_shelter.__file__)")   # now in site-packages!

### 11.3 The command-line program
Installing the package also created the **`shelter-report`** program declared in `[project.scripts]`.
From a **terminal** (with the same environment active) you can simply type:

```
shelter-report animals.csv
python -m animal_shelter animals.csv
```

Here we run both from the notebook. Note the warnings about the invalid lines: they are part of the output.

In [ ]:
csv_file = str(Path("animals.csv").resolve())        # full path: the new process runs in another folder

run([sys.executable, "-m", "animal_shelter", csv_file])     # runs animal_shelter/__main__.py

In [ ]:
import shutil
import sysconfig

program = shutil.which("shelter-report", path=sysconfig.get_path("scripts"))   # where pip put the program
print("The program created by pip:", program, "\n")

if program:
    run([program, csv_file])
    run([program, "missing.csv"])

---
## 12. Sharing and cleaning up

How can other people install your package? A few options, from the simplest:

| How | Command for the user |
|-----|----------------------|
| send them the `.whl` file | `pip install animal_shelter-0.1.0-py3-none-any.whl` |
| send them the project folder | `pip install ./shelter_project` |
| put the project on GitHub | `pip install git+https://github.com/<user>/<repo>.git` |
| publish it on **PyPI** (where `pip install numpy` downloads from) | `pip install animal-shelter` |

To publish on PyPI you create an account on [pypi.org](https://pypi.org) and upload the files in `dist/` with the tool `twine`
(`twine upload dist/*`). To practise, there is a separate test server: [test.pypi.org](https://test.pypi.org).
⚠️ Names on PyPI are unique: `animal-shelter` may already be taken!

Finally, let's uninstall our package from this environment:

In [ ]:
%pip uninstall -y animal-shelter

---
## 13. Summary

| Concept | What it is | Key points |
|---------|-----------|------------|
| **module** | a `.py` file | `import x`, `from x import y`, `import x as z` |
| module cache | `sys.modules` | a module runs only once; after editing it, `importlib.reload` or restart the kernel |
| `__name__ == "__main__"` | is the file run as a script? | code that should not run on import |
| `sys.path` | where Python looks for modules | current folder first: don't call your files `random.py`! |
| **package** | a folder with `__init__.py` | groups related modules |
| relative import | `from .exceptions import X` | import a module of the same package |
| `__init__.py` | the package's front door | choose the public names, `__version__`, `__all__` |
| `__main__.py` | `python -m package` | run a package as a program |
| `pyproject.toml` | the project's identity card | name, version, dependencies, scripts |
| `pip install -e .` | editable install | for developing: changes are visible without re-installing |
| `python -m build` | builds the distributions | a `.whl` (ready to install) and a `.tar.gz` (sources) |
| `pip install file.whl` | regular install | copies the package into `site-packages` |

### Golden rules
1. ✅ One module = one topic (animals, shelter, exceptions, ...)
2. ✅ Write docstrings for modules, classes and functions: `help()` shows them
3. ✅ Give your package a base exception class (`ShelterError`)
4. ✅ Libraries **raise** and **warn**; applications **catch** and **print**
5. ❌ Never fix imports by adding folders to `sys.path` by hand: **install** the package
6. ✅ Use the src layout and `pip install -e .` while developing
7. ✅ Keep the version in `pyproject.toml` and `__version__` equal, and increase it every time you share a new build

---
## 14. Exercises

**Exercise 1.** Add a `Bird(Animal)` class to `animals.py` (Exercise 1 of Lesson 1), with an extra attribute `can_fly` (default `True`).
Make `load_animals` accept lines with `kind` equal to `bird` and export `Bird` from `__init__.py` (don't forget `__all__`!).
Re-install in editable mode and check that `from animal_shelter import Bird` works from a new process.

**Exercise 2.** Create a new module `stats.py` with a function `average_age(shelter)`. It must raise a `ShelterError`
if the shelter is empty. Export it in `__init__.py`.

**Exercise 3.** Change the version to `0.2.0` (in **both** places), build the package again and install the new wheel.
Check the installed version with `%pip show animal-shelter`.
*Bonus*: `importlib.metadata.version("animal-shelter")` reads the version of an installed distribution. Could you use it in
`__init__.py` to avoid writing the version twice? What happens if the package is not installed?

**Exercise 4.** Make `shelter-report` accept **several** CSV files (`shelter-report a.csv b.csv`) and print one report for all of
them together (hint: `Shelter.__add__`). A missing file should be reported, without stopping the others.

**Exercise 5.** Try to run `python shelter_project/src/animal_shelter/shelter.py` from a terminal. Read the traceback (bottom up!):
why does it fail? What is the correct way to run code that lives inside a package?